# Ancilary Use Cases for VPEM and Cybersecurity Risk Management
 
There is significant value in using Neo4j for Vulnerability Prioritization and Exposure Management (VPEM) beyond the core use case of identifying internet-exposed vulnerabilities. Below are some additional queries and analyses that can be performed with vulnerability and KVE data.

In [27]:
from dotenv import load_dotenv
import os
from neo4j import GraphDatabase

load_dotenv()

# Connection details
URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
AUTH = (os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "password"))
DB = os.getenv("NEO4J_DB", "nvd")

# A helper method to run Cypher queries
def run_cypher(query, parameters=None):
    driver = GraphDatabase.driver(URI, auth=AUTH)
    try:
        with driver.session(database=DB) as session:
            result = session.run(query, parameters or {})
            # .data() converts the stream into a list of dictionaries
            return result.data() 
    finally:
        driver.close()

### The "Immediate Action" Overlap

This query identifies vulnerabilities that have both a Critical/High CVSS score and are Known to be Exploited in the wild. This represents the "top of the funnel" for any security program.

**Why it matters**: CISA KEV is a "source of truth" for what attackers are actually doing. If a CVE is in this list, the probability of an attack is near 100%.

In [28]:
import pandas as pd

query = """
MATCH (v:CVE)-[:LISTED_IN]->(cat:Catalog {name: 'CISA KEV'})
WHERE v.baseScore >= 7.0 
  AND date(v.kev_addedDate) >= date() - duration('P90D') 
RETURN v.id AS CVE, 
       v.baseScore AS Severity, 
       v.kev_addedDate AS Date_Added_to_KEV,
       v.kev_dueDate AS Remediation_Deadline,
       v.description AS Summary
ORDER BY v.kev_addedDate DESC
"""
results = run_cypher(query)
# Print the results as an html table
df = pd.DataFrame(results)
df

,CVE,Severity,Date_Added_to_KEV,Remediation_Deadline,Summary
0,CVE-2025-14847,7.5,2025-12-29,2026-01-19,Mismatched length fields in Zlib compressed pr...
1,CVE-2023-52163,8.8,2025-12-22,2026-01-12,Digiever DS-2105 Pro 3.1.0.71-11 devices allow...
2,CVE-2025-20393,10.0,2025-12-17,2025-12-24,Cisco is aware of a potential vulnerability.&n...
3,CVE-2025-59718,9.1,2025-12-16,2025-12-23,A improper verification of cryptographic signa...
4,CVE-2025-43529,8.8,2025-12-15,2026-01-05,A use-after-free issue was addressed with impr...
5,CVE-2025-14174,8.8,2025-12-12,2026-01-02,Out of bounds memory access in ANGLE in Google...
6,CVE-2018-4063,8.8,2025-12-12,2026-01-02,An exploitable remote code execution vulnerabi...
7,CVE-2025-58360,8.2,2025-12-11,2026-01-01,GeoServer is an open source server that allows...
8,CVE-2025-6218,7.8,2025-12-09,2025-12-30,RARLAB WinRAR Directory Traversal Remote Code ...
9,CVE-2025-62221,7.8,2025-12-09,2025-12-30,None


### Weakness Analysis (the CWE "Root Cause")

This query identifies which Common Weakness Enumerations (CWEs) are most frequently associated with exploited vulnerabilities. This helps you tell developers what coding patterns to stop using.

**Why it matters**: If "CWE-78: OS Command Injection" shows up as your #1 exploited weakness, you should prioritize a security training session or a linting rule specifically for that issue.

In [29]:
query = """
MATCH (w:CWE)<-[:HAS_PROBLEM_TYPE]-(v:CVE)-[:LISTED_IN]->(:Catalog {name: 'CISA KEV'})
RETURN w.id AS Weakness, 
       count(v) AS Exploited_CVE_Count
ORDER BY Exploited_CVE_Count DESC
LIMIT 10
"""
results = run_cypher(query)
# Print the results as an html table
df = pd.DataFrame(results)
df

,Weakness,Exploited_CVE_Count
0,CWE-78,36
1,CWE-502,26
2,CWE-416,24
3,CWE-20,21
4,CWE-22,20
5,CWE-94,18
6,CWE-306,14
7,CWE-284,14
8,CWE-122,14
9,CWE-288,12


### Vendor & Product Risk Concentration

Which vendors or products in your environment are "frequent flyers" in the KEV catalog ? This helps with **Vendor Risk Management** and **Supply Chain Security**.

**Why it matters**: This identifies "high-maintenance" products. If a specific VPN or Gateway product consistently appears in the KEV, it might be time to look for a more secure alternative.

In [30]:
query = """
MATCH (vend:Vendor)-[:PROVIDES]->(p:Product)<-[:AFFECTS]-(v:CVE)-[:LISTED_IN]->(:Catalog {name: 'CISA KEV'})
RETURN vend.name AS Vendor, 
       p.name AS Product, 
       count(DISTINCT v) AS KEV_Count
ORDER BY KEV_Count DESC
LIMIT 10
"""
results = run_cypher(query)
# Print the results as an html table
df = pd.DataFrame(results)
df

,Vendor,Product,KEV_Count
0,Microsoft,Windows Server 2019,115
1,Microsoft,Windows 10 Version 1809,108
2,Microsoft,Windows Server 2019 (Server Core installation),107
3,Microsoft,Windows Server 2016,105
4,Microsoft,Windows Server 2022,94
5,Microsoft,Windows Server 2016 (Server Core installation),93
6,Microsoft,Windows 10 Version 1607,93
7,Microsoft,Windows Server 2012 R2,89
8,Microsoft,Windows 10 Version 1507,88
9,Microsoft,Windows 10 Version 21H2,87


### The "Drop Everything" List: Recent Exploits on Live Assets

This query specifically looks for CVEs added to the KEV in the last 365 days that are currently running in your production environment on internet-facing instances.

This query is the "gold standard" for Exposure Management: It is the most valuable report a security team can receive:

- **Zeroing in on "Active Fire"**: A vulnerability scan might show 5,000 "Critical" bugs. This query likely narrows that down to 5 or 10. These are not theoretical risks; they are vulnerabilities that are currently being used by attackers and are sitting on your "front door" (public-facing servers).

- **Contextual Urgency**: By filtering for kev_addedDate within the last 90 days, you are focusing on the Exploitability Gap. This is the window between an exploit becoming public and an organization completing its patch cycle.

- **Direct Accountability**: Unlike a generic threat report, this provides the app.name and Instance_Name. You aren't just telling the team "Log4j is bad"; you are telling them "The Customer-Portal-API is currently vulnerable to an active exploit on server-prod-01."

In [31]:
query = """
MATCH (v:CVE)-[:LISTED_IN]->(cat:Catalog {name: 'CISA KEV'})
MATCH (v)-[:IDENTIFIED_IN]->(l:Library)-[:DEPENDENCY_OF]->(ba:BuildArtifact)-[:RUNNING_AS]->(app:Application)-[:HOSTED_ON]->(ins:ComputeInstance)

// Filter for internet reachability
WHERE ins.public_ip IS NOT NULL 
// Filter for recent KEV additions (365 days)
  AND date(v.kev_addedDate) >= date() - duration('P365D')

RETURN v.id AS Recent_CVE, 
       v.kev_addedDate AS KEV_Date, 
       app.name AS Affected_App, 
       ins.name AS Exposed_Instance, 
       ins.public_ip AS IP_Address
ORDER BY v.kev_addedDate DESC
"""
results = run_cypher(query)
# Print the results as an html table
df = pd.DataFrame(results)
df

,Recent_CVE,KEV_Date,Affected_App,Exposed_Instance,IP_Address
0,CVE-2023-52163,2025-12-22,EdgeAuthenticator,edge-auth-01,52.14.99.1
1,CVE-2025-24990,2025-10-14,EdgeAuthenticator,edge-auth-01,52.14.99.1
